# Class 3 - Structured Output & Prompt Debugging

**Week 4: Prompt Engineering (Basic to Intermediate)**

### Learning objectives
By the end of this notebook you will be able to:
- Explain why structured output matters once a prompt's response feeds into more code.
- Write a prompt that reliably returns valid JSON, and know why "as JSON" alone is not enough.
- Use a JSON-schema-constrained output mode where your provider supports it.
- Validate model output with `json.loads` and build a simple retry-on-failure loop.
- Apply a repeatable process to debug a prompt that keeps failing.


## Setup

Run this cell first. It reads your API key from the environment - never hardcode a key in a notebook.

Set one of these in your shell before launching Jupyter:

```bash
export OPENAI_API_KEY="sk-..."
# or
export ANTHROPIC_API_KEY="sk-ant-..."
```

You'll also need the matching SDK installed:

```bash
pip install openai anthropic
```


In [ ]:
import os

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY")
ANTHROPIC_API_KEY = os.environ.get("ANTHROPIC_API_KEY")

if not OPENAI_API_KEY and not ANTHROPIC_API_KEY:
    print(
        "No API key found in the environment.\n"
        "Set one of these before running the demo cells below:\n"
        "  export OPENAI_API_KEY='sk-...'\n"
        "  export ANTHROPIC_API_KEY='sk-ant-...'\n"
        "The code below will run as soon as a key is set - nothing is hardcoded here."
    )
else:
    provider = "OpenAI" if OPENAI_API_KEY else "Anthropic"
    print(f"Found an API key for {provider}. You're ready to run the demo cells below.")

# --- Minimal provider-agnostic chat helper -------------------------------
# Uses OpenAI if OPENAI_API_KEY is set, otherwise falls back to Anthropic.

def ask(system_prompt, user_prompt, model=None, temperature=0.7, max_tokens=400):
    """Send a system + user prompt to whichever provider has a key set.
    Returns the assistant's text reply as a plain string."""
    if OPENAI_API_KEY:
        from openai import OpenAI
        client = OpenAI(api_key=OPENAI_API_KEY)
        model = model or "gpt-4o-mini"
        resp = client.chat.completions.create(
            model=model,
            temperature=temperature,
            max_tokens=max_tokens,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ],
        )
        return resp.choices[0].message.content
    elif ANTHROPIC_API_KEY:
        import anthropic
        client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
        model = model or "claude-3-5-haiku-20241022"
        resp = client.messages.create(
            model=model,
            max_tokens=max_tokens,
            temperature=temperature,
            system=system_prompt,
            messages=[{"role": "user", "content": user_prompt}],
        )
        return resp.content[0].text
    else:
        raise RuntimeError(
            "No API key set. Set OPENAI_API_KEY or ANTHROPIC_API_KEY in your environment first."
        )


## Concept 1 - Requesting JSON

"Give me JSON" leaves the exact shape up to the model. Naming the keys, types, and saying "ONLY JSON,
no other text" removes almost all of the ambiguity.

In [ ]:
import json

weak_prompt = (
    "Give me the product details as JSON for: 24oz stainless steel water bottle, "
    "$34.99, currently in stock."
)
strong_prompt = (
    'Return ONLY valid JSON, no other text, no markdown code fences, matching exactly this shape: '
    '{"name": string, "price_usd": number, "in_stock": boolean}.\n\n'
    "Product: 24oz stainless steel water bottle, $34.99, currently in stock."
)

for label, prompt in [("WEAK", weak_prompt), ("STRONG", strong_prompt)]:
    print(f"--- {label} ---")
    raw = ask("You are a data extraction assistant.", prompt, temperature=0)
    print(raw)
    print()


## Concept 2 - Schema-Constrained Output

Where the API supports it, a JSON-schema mode enforces the shape at the token level instead of relying
on the model choosing to follow instructions. This demo is OpenAI-specific (`response_format`); it's
skipped automatically if you're using an Anthropic key instead.

In [ ]:
def ask_json_schema(system_prompt, user_prompt, schema, model="gpt-4o-mini"):
    """Requires OPENAI_API_KEY. Uses OpenAI's structured-output (json_schema) mode
    so the response is guaranteed to match `schema`."""
    if not OPENAI_API_KEY:
        raise RuntimeError(
            "This demo requires OPENAI_API_KEY (json_schema mode is an OpenAI-specific feature)."
        )
    from openai import OpenAI
    client = OpenAI(api_key=OPENAI_API_KEY)
    resp = client.chat.completions.create(
        model=model,
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": user_prompt},
        ],
        response_format={
            "type": "json_schema",
            "json_schema": {
                "name": "product",
                "schema": schema,
                "strict": True,
            },
        },
    )
    return resp.choices[0].message.content


product_schema = {
    "type": "object",
    "properties": {
        "name": {"type": "string"},
        "price_usd": {"type": "number"},
        "in_stock": {"type": "boolean"},
    },
    "required": ["name", "price_usd", "in_stock"],
    "additionalProperties": False,
}

if OPENAI_API_KEY:
    raw = ask_json_schema(
        "You are a data extraction assistant.",
        "Product: 24oz stainless steel water bottle, $34.99, currently in stock.",
        product_schema,
    )
    print(raw)
    print(json.loads(raw))
else:
    print("Skipping - set OPENAI_API_KEY to run this schema-constrained demo.")


## Concept 3 - Validating What You Get Back

Never trust model output blindly. This helper parses the JSON, checks required keys, and retries once
with a corrective follow-up prompt if anything is wrong.

In [ ]:
def get_validated_json(system_prompt, user_prompt, required_keys, max_retries=1):
    """Calls the model, tries to parse JSON, and retries once with a corrective
    prompt if parsing fails or required keys are missing."""
    prompt = user_prompt
    for attempt in range(max_retries + 1):
        raw = ask(system_prompt, prompt, temperature=0)
        try:
            data = json.loads(raw)
            missing = [k for k in required_keys if k not in data]
            if missing:
                raise ValueError(f"Missing keys: {missing}")
            return data
        except (json.JSONDecodeError, ValueError) as e:
            print(f"Attempt {attempt + 1} failed: {e}")
            prompt = (
                f"{user_prompt}\n\nYour previous response was not valid: {e}. "
                "Return ONLY the corrected JSON, nothing else."
            )
    raise RuntimeError("Could not get valid JSON after retries.")


result = get_validated_json(
    "You are a data extraction assistant.",
    'Return ONLY JSON: {"name": string, "price_usd": number, "in_stock": boolean} '
    "for: 24oz stainless steel water bottle, $34.99, in stock.",
    required_keys=["name", "price_usd", "in_stock"],
)
print(result)


## Concept 4 - Debugging a Failing Prompt

This mirrors the "Workshop: From Failing to Reliable" slide. The same task - extract 3 skills from a
resume as JSON - is attempted three times, each attempt tightening the prompt based on what broke last time.

In [ ]:
resume_snippet = (
    "Experienced backend engineer skilled in Python, distributed systems, and "
    "mentoring. Led a team of 4 to migrate a monolith to microservices. "
    "Comfortable with PostgreSQL, Kafka, and Docker."
)

attempt_1 = f"Give me the top 3 skills from this resume as JSON.\n\n{resume_snippet}"
attempt_2 = (
    'Return ONLY JSON: {"skills": [string, string, string]} - exactly 3 skills, '
    f"no other text.\n\n{resume_snippet}"
)

print("--- Attempt 1 (often has extra prose around the JSON) ---")
print(ask("You are a resume parser.", attempt_1, temperature=0))

print("\n--- Attempt 2 (tighter instruction, still not schema-enforced) ---")
print(ask("You are a resume parser.", attempt_2, temperature=0))

print("\n--- Attempt 3 (schema-enforced, requires OPENAI_API_KEY) ---")
skills_schema = {
    "type": "object",
    "properties": {
        "skills": {
            "type": "array",
            "items": {"type": "string"},
            "minItems": 3,
            "maxItems": 3,
        }
    },
    "required": ["skills"],
    "additionalProperties": False,
}
if OPENAI_API_KEY:
    raw = ask_json_schema(
        "You are a resume parser.",
        f"Extract exactly 3 skills.\n\n{resume_snippet}",
        skills_schema,
    )
    print(raw)
else:
    print("Skipping - set OPENAI_API_KEY to run the schema-enforced version.")


## Challenges

Work through these on your own. No solutions are provided - write your own prompts and code below each prompt.


### Challenge 1 - Fix the Fence

The prompt below reliably comes back wrapped in a ```json ... ``` markdown fence, which breaks
`json.loads`. Fix it two ways: (a) rewrite the prompt to forbid markdown, and (b) write a small Python
function that strips fences before parsing, as a safety net.

**Acceptance criteria:** `json.loads` succeeds on the final result, using at least one of the two fixes.

In [ ]:
fence_prone_prompt = (
    "Give me a JSON object with the keys name and price_usd for: a 24oz water bottle at $34.99."
)

# TODO: (a) rewrite fence_prone_prompt to forbid markdown formatting, and/or
# TODO: (b) write a function that strips ```json fences from a string before json.loads().


### Challenge 2 - Validate & Retry

Extend `get_validated_json` (or write your own version) so it also checks that `price_usd` is a
positive number, not just present - and retries with a corrective prompt if it isn't.

**Acceptance criteria:** the validator rejects a response where `price_usd` is missing, non-numeric, or negative.

In [ ]:
# TODO: write a validation function that checks price_usd is present, numeric, and > 0.
def get_validated_product(system_prompt, user_prompt, max_retries=1):
    pass


### Challenge 3 - Enforce a Schema

Design your own JSON schema for a different object - for example a `recipe` with `name`, `servings`,
and an `ingredients` list of strings. Use `ask_json_schema` to extract it from a short description you write.

**Acceptance criteria:** a schema dict, a short input description, and a successfully parsed JSON result.

In [ ]:
# TODO: define your own schema (e.g. a recipe) and call ask_json_schema() to extract it.
my_schema = {}
my_description = ""


### Challenge 4 - Diagnose a Bug

The prompt below is intended to always return exactly 3 tags as JSON, but sometimes returns 2, 4, or
adds a `"confidence"` key nobody asked for. Apply the 6-step debugging process from Concept 4
(isolate, reproduce, compare, localize, patch, re-test) and write your findings in a markdown cell,
then apply your fix in code.

**Acceptance criteria:** a written diagnosis of the likely cause, plus a revised prompt that fixes it.

In [ ]:
buggy_prompt = (
    "Tag this article with relevant topics as JSON: "
    "'A new study shows coral reefs recovering faster than expected near marine-protected zones.'"
)

# TODO: run buggy_prompt a few times, observe the failure pattern, and write a fixed version.


### Challenge 5 - Break It on Purpose

Write a prompt you predict will fail to reliably produce valid JSON (think about what you learned in
Concept 1 and the failure-patterns slide). Run it 3 times and explain, in a comment or markdown cell,
why it failed the way you expected.

**Acceptance criteria:** a prompt, 3 runs of output, and a written explanation of the failure pattern observed.

In [ ]:
# TODO: write a prompt you expect to fail, run it 3 times, and explain the failure pattern you predicted.
my_prompt_designed_to_fail = ""
